# BiLSTM + GP pre eventovy multi-horizon experiment

Notebook pouziva modul `bilstm_gp_multihorizon_experiment.py` a spusta novy eventovy experiment nad `event_omni_prepared.csv`.

In [ ]:
from bilstm_gp_multihorizon_experiment import (
    ExperimentConfig,
    build_event_metadata,
    default_paths,
    load_event_dataset,
    run_full_grid,
    set_seed,
    split_events,
)

import pandas as pd

set_seed()
paths = default_paths()
df = load_event_dataset(paths["data"])
event_meta = build_event_metadata(df)
train_meta, val_meta, test_meta = split_events(event_meta)

train_meta.to_csv(paths["splits"] / "train_events.csv", index=False)
val_meta.to_csv(paths["splits"] / "val_events.csv", index=False)
test_meta.to_csv(paths["splits"] / "test_events.csv", index=False)

print(f"Rows: {len(df):,}")
print(f"Events: {df['event_no'].nunique():,}")
display(df.head())

In [ ]:
split_summary = [
    {"split": "train", "events": len(train_meta), "rows": int(df[df['event_no'].isin(train_meta['event_no'])].shape[0])},
    {"split": "val", "events": len(val_meta), "rows": int(df[df['event_no'].isin(val_meta['event_no'])].shape[0])},
    {"split": "test", "events": len(test_meta), "rows": int(df[df['event_no'].isin(test_meta['event_no'])].shape[0])},
]
display(split_summary)
split_labels = pd.concat([
    train_meta[['event_no']].assign(split='train'),
    val_meta[['event_no']].assign(split='val'),
    test_meta[['event_no']].assign(split='test'),
], ignore_index=True)
display(
    event_meta.merge(split_labels, on='event_no', how='left')[['split', 'severity_bin']]
    .value_counts()
    .rename('events')
    .reset_index()
)


In [ ]:
config = ExperimentConfig(
    lookbacks=[6, 12, 18, 24],
    feature_keys=["dst_only", "dst_v", "dst_bz", "dst_bz_v"],
    thresholds=[-20.0, -50.0],
    epochs=50,
    batch_size=256,
    hidden=64,
    dropout=0.2,
    patience=8,
    max_gp_points=4000,
    gp_restarts=1,
)

regression_df, classification_df, overview_df = run_full_grid(
    df_events=df,
    train_meta=train_meta,
    val_meta=val_meta,
    test_meta=test_meta,
    config=config,
    paths=paths,
)

display(regression_df.head())
display(classification_df.head())
display(overview_df.head())

In [ ]:
best_regression = (
    regression_df[regression_df['model'] == 'bilstm_gp']
    .sort_values(['horizon_hours', 'rmse', 'mae'], ascending=[True, True, True])
    .groupby('horizon_hours')
    .head(3)
    .reset_index(drop=True)
)

best_classification = (
    classification_df[
        (classification_df['model'] == 'bilstm_gp')
        & (classification_df['dst_threshold'] == -20.0)
    ]
    .sort_values(['horizon_hours', 'f1_class1', 'recall_class1'], ascending=[True, False, False])
    .groupby('horizon_hours')
    .head(3)
    .reset_index(drop=True)
)

display(best_regression)
display(best_classification)